In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct
import astropy.io.fits as fits
import tools21cm as t2c
from sklearn.decomposition import FastICA
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import matplotlib
matplotlib.style.use('/home/ppxjf3/paper_params.mplstyle') 


from astropy.cosmology import Planck13 as cosmoP
from astropy.cosmology import FlatLambdaCDM, LambdaCDM

import astropy.units as u
cosmo = FlatLambdaCDM(H0=71 * u.km / u.s / u.Mpc, Om0=0.27)


In [ ]:
import matplotlib.font_manager
import matplotlib as mpl

column_width=240# call "\the\columnwidth" in LaTeX to find
ppi=72#default ppi, can be left the same

scale=2
fig_width=column_width/ppi*scale#inches
fig_height=3*scale#inches

##SET FONT SIZES
font_small_size = 9
font_medium_size = 12
font_bigger_size =14

plt.rc('font', size=font_small_size) # controls default text sizes
plt.rc('axes', titlesize=font_small_size) # fontsize of the axes title
plt.rc('axes', labelsize=font_medium_size) # fontsize of the x and y labels
plt.rc('xtick', labelsize=font_bigger_size) # fontsize of the tick labels
plt.rc('ytick', labelsize=font_bigger_size) # fontsize of the tick labels
plt.rc('legend', fontsize=font_small_size) # legend fontsize
plt.rc('figure', titlesize=font_bigger_size)

#DPI of MNRAS is 300
mpl.rcParams['figure.dpi'] = 300/scale

Define functions

In [ ]:
def pearson_correl(x,y):
 return (np.sum((x-np.mean(x))*(y-np.mean(y))))/(np.sqrt(np.sum((x-np.mean(x))**2)*np.sum((y-np.mean(y))**2)))

def do_fastica(data, comps):
    shape = data.shape
    f_ica = FastICA(n_components=comps)
    #generate the 4 componets
    S = f_ica.fit_transform(data.reshape((shape[0]*shape[1],shape[2])))
    
    #get mixing matrix
    A = f_ica.mixing_
    
    #make model
    model_fICA = (np.matmul(A,S.T).T).reshape((shape[0],shape[1],shape[2]))
    
    #get resids
    resids_fICA = data - model_fICA #residuals 
    
    return model_fICA, resids_fICA

def zFromNu(nu):
    """
    Convert frequency of 21cm line to redshift
    
    Input: nu [MHz]
    """
    nu21 = 1.420405e3  #MHz
    return nu21/nu - 1.0

def NuFromz(z):
    """
    Convert frequency of 21cm line to redshift
    
    Input: nu [MHz]
    """
    nu21 = 1.420405e3  #MHz
    return nu21/(z + 1.0)

def correlation_graph(LC, RSD_LC, res_LC, res_RSD, freq, name):
    cc_LC = np.empty([res_LC.shape[2]])
    cc_RSD = np.empty([res_RSD.shape[2]])

    for ii in range(0,res_RSD.shape[2]):
        cc_LC[ii] = pearson_correl(res_LC[:,:,ii],LC[:,:,ii])
        cc_RSD[ii] = pearson_correl(res_RSD[:,:,ii],RSD_LC[:,:,ii])
    
    plt.plot(freq, cc_RSD, label = 'RSD')
    plt.plot(freq, cc_LC, label = 'LC')
    plt.ylabel('correlation')
    plt.xlabel('freq')
    plt.legend()
    plt.savefig('/home/ppxjf3/RSD_LC/comparison/graphs/' + name, dpi=330)
    plt.show()
    
    plt.plot(freq, cc_RSD - cc_LC)
    plt.ylabel('residual (cc_RSD - cc_LC)')
    plt.xlabel('freq')
    plt.show()
    
def get_lengths(nu_low, nu_hi, nu_mid, theta_FOV):
    z_lo = zFromNu(nu_low)
    z_mid = zFromNu(nu_mid)
    z_hi = zFromNu(nu_hi)
    
    L_para = cosmo.comoving_distance(z_lo) - cosmo.comoving_distance(z_hi)
    L_perp = cosmo.comoving_distance(z_mid) * (np.pi * theta_FOV / 180.0)
    return L_para/u.Mpc, L_perp/u.Mpc

Open lightcones

In [ ]:
base = '/home/ppxjf3/RSD_LC/comparison/FINAL_LIGHTCONES_PEC_VEL_PAPER/'
npix=512
nfreq=401

with open(base + 'LightconeRSD_N512_FOV1.0000_dnu0.10MHz_180.00MHz_140.00MHz_ds0.005991_div00.00_pv1_oneevent0_evo1_lcon0_dz_000.10.dat', "rb") as fid:
    # Read the binary data from the file
    data = fid.read()

    # Unpack the binary data into a tuple of floats
    RSD = struct.unpack('f' * (len(data) // struct.calcsize('f')), data)

RSD_z01_nu01_1deg = np.array(RSD)*1e3
RSD_z01_nu01_1deg = RSD_z01_nu01_1deg.reshape(npix,npix,nfreq)
        
with open(base + 'LightconeRSD_N512_FOV1.0000_dnu0.10MHz_180.00MHz_140.00MHz_ds0.014978_div00.00_pv1_oneevent0_evo1_lcon1_dz_000.10.dat', "rb") as fid:
    # Read the binary data from the file
    data = fid.read()

    # Unpack the binary data into a tuple of floats
    LC = struct.unpack('f' * (len(data) // struct.calcsize('f')), data)
LC_z01_nu01_1deg = np.array(LC)*1e3
LC_z01_nu01_1deg = LC_z01_nu01_1deg.reshape(npix,npix,nfreq)


Make Figure 1 of paper, to visualise the difference between the two methods

In [ ]:
import matplotlib.gridspec as gridspec

idx=150
together = np.vstack([RSD_z01_nu01_1deg[:,:,idx],LC_z01_nu01_1deg[:,:,idx]])
vmin = np.min(together)
vmax = np.max(LC_z01_nu01_1deg[:,:,idx])

fig = plt.figure(figsize=(fig_width*2, fig_height*1.75))
gs = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.0)

# Top plot
ax0 = fig.add_subplot(gs[0,0])
ax1 = fig.add_subplot(gs[0,1])
ax0.imshow(RSD_z01_nu01_1deg[:,:,idx], vmin=vmin, vmax=vmax, extent=[0,1.0,1.0,0], cmap='inferno')#, norm='log')
c = ax1.imshow(LC_z01_nu01_1deg[:,:,idx],vmin=vmin, vmax=vmax, extent=[0,1.0,1.0,0], cmap='inferno')#, norm='log')
ax0.set_xlabel(r'degrees', fontsize = 14)
ax0.set_ylabel(r'degrees', fontsize = 14)
ax1.set_xlabel(r'degrees', fontsize = 14)
ax1.set_yticklabels([])
#ax1.set_ylabel(r'degrees', fontsize = 16)

# Bottom plot
ax2 = fig.add_subplot(gs[1,:])
pmc_lc_cd = ax2.imshow(RSD_z01_nu01_1deg[0,:,:], extent=[140,180,0,1.0], vmin=vmin, vmax=vmax, aspect='auto', cmap='inferno')
ax2.set_xlabel(r'$\nu /\rm MHz$', fontsize = 14)
ax2.set_ylabel(r'degrees', fontsize = 14)

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.85, 0.075, 0.03, 0.71])
cb = fig.colorbar(c, cax=cbar_ax)
cb.set_label(label = r'$ \delta T_B \rm [mK]$', fontsize = 14)
# Save with tight layout
fig.savefig(base + 'graphs_for_paper/look_at_RSD.pdf', dpi=330)
plt.show()

In [ ]:
freq = np.arange(140, 180.1, 0.1)

z = np.arange(6.0, 10.0, 0.1)

nu = NuFromz(z)

In [ ]:
base = '/home/ppxjf3/RSD_LC/comparison/FINAL_LIGHTCONES_PEC_VEL_PAPER/'
with fits.open(base + 'Fg_N512_FOV1.0_140_180.0MHz_0.1MHz.fits', memmap=True) as hdu:
    FG = np.array(hdu[0].data)  
    
FG=FG*1e3
fig, ax = plt.subplots(1,1)
pmc_lc_cd = ax.imshow(FG[:,:,100])
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(pmc_lc_cd, cax=cax, orientation='vertical', label='mK')

In [ ]:
FG_LC = FG + LC_z01_nu01_1deg
FG_RSD = FG + RSD_z01_nu01_1deg

c = 4
freq = np.arange(140, 180.1, 0.1)
model1, resids_LC_FG = do_fastica(FG_LC, c)
model2, resids_RSD_FG = do_fastica(FG_RSD, c)


In [ ]:
kbins=15
mubins=15

  
i0 = 0
i1 = 101  

L_para, L_perp = get_lengths(freq[i0], freq[i1], (freq[i0]+freq[i1])/2, 1.0)
box_dims = [L_perp, L_perp, L_para]


p_lc_eor_res, k_lc_eor_res, n_lc_res = t2c.power_spectrum_1d(resids_LC_FG[:,:,i0:i1], kbins=kbins, box_dims=box_dims, binning =  'log', return_n_modes=True)
p_rsd_lc_eor_res, k_rsd_lc_eor_res, n_rsd_res = t2c.power_spectrum_1d(resids_RSD_FG[:,:,i0:i1], box_dims=box_dims, kbins=kbins, binning =  'log', return_n_modes=True)
p_lc_eor, k_lc_eor, n_lc = t2c.power_spectrum_1d(LC_z01_nu01_1deg[:,:,i0:i1], kbins=kbins, box_dims=box_dims, binning =  'log', return_n_modes=True)
p_rsd, k_rsd, n_rsd = t2c.power_spectrum_1d(RSD_z01_nu01_1deg[:,:,i0:i1], box_dims=box_dims, kbins=kbins, binning =  'log', return_n_modes=True)



In [ ]:
d_lc = (p_lc_eor*k_lc_eor**3)/(2*np.pi**2)
d_rsd = (p_rsd*k_rsd**3)/(2*np.pi**2)
d_lc_res = (p_lc_eor_res*k_lc_eor_res**3)/(2*np.pi**2)
d_rsd_res = (p_rsd_lc_eor_res*k_rsd_lc_eor_res**3)/(2*np.pi**2)

err_lc_res= np.empty_like(n_lc_res)
err_rsd_res= np.empty_like(n_rsd_res)

for ii in range(0,n_rsd_res.shape[0]):
    err_lc_res[ii] = d_lc_res[ii]/np.sqrt(n_lc_res[ii])
        
    err_rsd_res[ii] = d_rsd_res[ii]/np.sqrt(n_rsd_res[ii])


In [ ]:
noise = np.loadtxt("/home/ppxjf3/RSD_LC/eor_sensitivity_full_baselines_AAstar.txt")
print(noise.shape)
xx_AAstar = noise[0,:]
sense1d_AAstar = noise[1,:]

noise_AA4 = np.loadtxt("/home/ppxjf3/RSD_LC/eor_sensitivity_full_baselines_AA4.txt")
xx_AA4 = noise_AA4[0,:]
sense1d_AA4 = noise_AA4[1,:]

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(fig_width, fig_height))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

# Top plot
ax1 = fig.add_subplot(gs[0])
ax1.plot(k_lc_eor, d_lc, color='k', label=r'$\rm basic$')
ax1.plot(k_rsd, d_rsd, color='r', label=r'$\rm extended$')
ax1.errorbar(k_lc_eor_res, d_lc_res, err_lc_res, color='k', ls='none', capsize=2.5, marker='x')
ax1.errorbar(k_rsd_lc_eor_res, d_rsd_res, err_rsd_res, color='r', ls='none', capsize=2.5, marker='x')
"""ax1.fill_between(xx, sense1d_both, 0, color='grey', alpha=0.5)#s label='AA* Sensitivity')
"""
ax1.fill_between(xx_AAstar, sense1d_AAstar, 0, color='grey', alpha=0.5, label='AA* Sensitivity')
ax1.fill_between(xx_AA4, sense1d_AA4, 0, color='pink', alpha=0.5, label='AA4 Sensitivity')
ax1.set_yscale('log')
ax1.set_xscale('log')
ax1.set_xlim(0.07, 10)
ax1.set_ylim(1, 10000)
ax1.set_ylabel(r'$\Delta_{\rm 3D} ^2(k)/\rm mK^2$', fontsize=16)
ax1.legend(frameon=False, fontsize=14)
ax1.tick_params(labelbottom=False)  # Hide x labels on top panel


# Bottom plot
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(k_lc_eor, d_rsd / d_lc, 'r')
ax2.plot(k_lc_eor, d_lc / d_lc, 'k', linestyle='dashed')
ax2.set_xscale('log')
#ax2.set_xlim(0.07, 4)
#ax2.set_ylim(0.,2.4)
ax2.set_xlabel(r'$k/(\rm Mpc)^{-1}$', fontsize=16)
ax2.set_ylabel(r'$\frac{\Delta_{\rm 3D, full}^2}{\Delta_{\rm 3D, std}^2}$', fontsize=16)

# Save with tight layout
fig.tight_layout()
fig.savefig(base + 'graphs_for_paper/3d_ps_fg_removal_full_baseline.pdf', dpi=330)
plt.show()

In [ ]:
from scipy.interpolate import interp1d
from scipy.optimize import brentq
# Interpolation functions
x1 = xx_AA4
x2 = k_rsd
f1 = interp1d( xx_AA4, sense1d_AA4, kind='linear')
f2 = interp1d(k_rsd, d_rsd, kind='linear')

# Function representing the difference between the lines
def diff(x):
    return f1(x) - f2(x)

# Find a sign change in the difference (i.e., an intersection)
x_common = np.linspace(max(x1[0], x2[0]), min(x1[-1], x2[-1]), 1000)
sign_change_indices = np.where(np.diff(np.sign(diff(x_common))))[0]

if len(sign_change_indices) > 0:
    # Use the first sign change interval for root finding
    i = sign_change_indices[0]
    x_intersect = brentq(diff, x_common[i], x_common[i + 1])
    y_intersect = f1(x_intersect)

    print(f"Intersection at x = {x_intersect:.4f}, y = {y_intersect:.4f}")
else:
    print("No intersection found in the given range.")

And for the Cosmic Dawn

In [ ]:
#cosmic dawm
nfreq=201
#with open(base + 'LightconeRSD_N512_FOV1.0000_dnu0.10MHz_095.00MHz_075.00MHz_ds0.002976_div00.00_pv1_oneevent0_evo1_lcon0_dz_000.10.dat', "rb") as fid:
with open(base+'LightconeRSD_N512_FOV1.0000_dnu0.10MHz_095.00MHz_075.00MHz_ds0.003099_div00.00_pv1_oneevent0_evo1_lcon0_dz_000.10.dat', "rb") as fid:
    # Read the binary data from the file
    data = fid.read()

    # Unpack the binary data into a tuple of floats
    RSD = struct.unpack('f' * (len(data) // struct.calcsize('f')), data)

RSD_z01_nu01_1deg = np.array(RSD)*1e3
RSD_z01_nu01_1deg = RSD_z01_nu01_1deg.reshape(npix,npix,nfreq)
        
with open(base +"LightconeRSD_N512_FOV1.0000_dnu0.10MHz_095.00MHz_075.00MHz_ds0.007747_div00.00_pv1_oneevent0_evo1_lcon1_dz_000.10.dat", "rb") as fid:
    # Read the binary data from the file
    data = fid.read()

    # Unpack the binary data into a tuple of floats
    LC = struct.unpack('f' * (len(data) // struct.calcsize('f')), data)

LC_z01_nu01_1deg = np.array(LC)*1e3
LC_z01_nu01_1deg = LC_z01_nu01_1deg.reshape(npix,npix,nfreq)

with fits.open(base + 'Fg_N512_FOV1.0_75_95.0MHz_0.1MHz.fits', memmap=True) as hdu:
    FG = np.array(hdu[0].data)  
    
FG=FG*1e3

In [ ]:
FG_LC = FG + LC_z01_nu01_1deg
FG_RSD = FG + RSD_z01_nu01_1deg

freq01 = np.arange(75, 95.1, 0.1)

c = 4
model1, resids_LC_FG = do_fastica(FG_LC, c)
model2, resids_RSD_FG = do_fastica(FG_RSD, c)

In [ ]:
kbins=12
mubins=15

i0 = 50
i1 = 151

L_para, L_perp = get_lengths(freq01[i0], freq01[i1], (freq01[i0]+freq01[i1])/2, 1.0)
box_dims = [L_perp, L_perp, L_para]


p_lc_eor_res, k_lc_eor_res, n_lc_res = t2c.power_spectrum_1d(resids_LC_FG[:,:,i0:i1], kbins=kbins, box_dims=box_dims, binning =  'log', return_n_modes=True)
p_rsd_lc_eor_res, k_rsd_lc_eor_res, n_rsd_res = t2c.power_spectrum_1d(resids_RSD_FG[:,:,i0:i1], box_dims=box_dims, kbins=kbins, binning =  'log', return_n_modes=True)
p_lc_eor, k_lc_eor, n_lc = t2c.power_spectrum_1d(LC_z01_nu01_1deg[:,:,i0:i1], kbins=kbins, box_dims=box_dims, binning =  'log', return_n_modes=True)
p_rsd, k_rsd, n_rsd = t2c.power_spectrum_1d(RSD_z01_nu01_1deg[:,:,i0:i1], box_dims=box_dims, kbins=kbins, binning =  'log', return_n_modes=True)


In [ ]:
d_rsd = (p_rsd*k_rsd**3)/(2*np.pi**2)
d_lc = (p_lc_eor*k_lc_eor**3)/(2*np.pi**2)
d_lc_res = (p_lc_eor_res*k_lc_eor_res**3)/(2*np.pi**2)
d_rsd_res = (p_rsd_lc_eor_res*k_rsd_lc_eor_res**3)/(2*np.pi**2)


#calculate errors
err_lc_res= np.empty_like(n_lc_res)
err_rsd_res= np.empty_like(n_lc_res)

for ii in range(0,n_lc_res.shape[0]):
    err_lc_res[ii] = d_lc_res[ii]/np.sqrt(n_lc_res[ii])
        
    err_rsd_res[ii] = d_rsd_res[ii]/np.sqrt(n_rsd_res[ii])

In [ ]:
noise = np.loadtxt("/home/ppxjf3/RSD_LC/cd_sensitivity_full_baselines_AAstar.txt")
print(noise.shape)
xx_AAstar = noise[0,:]
sense1d_AAstar = noise[1,:]

noise_AA4 = np.loadtxt("/home/ppxjf3/RSD_LC/cd_sensitivity_full_baselines_AA4.txt")
xx_AA4 = noise_AA4[0,:]
sense1d_AA4 = noise_AA4[1,:]

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(fig_width, fig_height))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

# Top plot
ax1 = fig.add_subplot(gs[0])
ax1.plot(k_lc_eor, d_lc, color='k', label=r'$\delta T_b(\rm 21cm)$')
ax1.plot(k_rsd, d_rsd, color='b')
ax1.errorbar(k_lc_eor_res, d_lc_res, err_lc_res, color='k', ls='none', capsize=2.5, marker='x', label=r'$\delta T_b(\rm reconstructed)$')
ax1.errorbar(k_rsd_lc_eor_res, d_rsd_res, err_rsd_res, color='b', ls='none', capsize=2.5, marker='x')
#ax1.fill_between(xx*0.71, sense1d_both, 0, color='grey', alpha=0.5, label='21cmSense Sensitivity')
ax1.fill_between(xx_AAstar, sense1d_AAstar, 0, color='grey', alpha=0.5, label='AA* Sensitivity')
ax1.fill_between(xx_AA4, sense1d_AA4, 0, color='cyan', alpha=0.5, label='AA4 Sensitivity')
ax1.set_yscale('log')
ax1.set_xscale('log')
ax1.set_xlim(0.07, 7.5)
ax1.set_ylim(10, 10000)
ax1.set_ylabel(r'$\Delta_{\rm 3D} ^2(k)/\rm mK^2$', fontsize=16)
ax1.legend(frameon=False, fontsize=14)
ax1.tick_params(labelbottom=False)  # Hide x labels on top panel

# Bottom plot
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(k_lc_eor, d_rsd / d_lc, 'b')
ax2.plot(k_lc_eor, d_lc / d_lc, 'k', linestyle='dashed')
ax2.set_xscale('log')
ax1.set_xlim(0.07, 7.5)
ax2.set_ylim(0.,2.1)
ax2.set_xlabel(r'$k/(\rm Mpc)^{-1}$', fontsize=16)
ax2.set_ylabel(r'$\frac{\Delta_{\rm 3D, full}^2}{\Delta_{\rm 3D, std}^2}$', fontsize=16)

# Save with tight layout
fig.tight_layout()
fig.savefig(base + 'graphs_for_paper/cd_3d_ps_fg_removal.pdf', dpi=330)
plt.show()

In [ ]:
from scipy.interpolate import interp1d
from scipy.optimize import brentq
# Interpolation functions
x1 = xx_AAstar
x2 = k_rsd
f1 = interp1d(xx_AAstar, sense1d_AAstar, kind='linear')
f2 = interp1d(k_rsd, d_rsd, kind='linear')

# Function representing the difference between the lines
def diff(x):
    return f1(x) - f2(x)

# Find a sign change in the difference (i.e., an intersection)
x_common = np.linspace(max(x1[0], x2[0]), min(x1[-1], x2[-1]), 1000)
sign_change_indices = np.where(np.diff(np.sign(diff(x_common))))[0]

if len(sign_change_indices) > 0:
    # Use the first sign change interval for root finding
    i = sign_change_indices[0]
    x_intersect = brentq(diff, x_common[i], x_common[i + 1])
    y_intersect = f1(x_intersect)

    print(f"Intersection at x = {x_intersect:.4f}, y = {y_intersect:.4f}")
else:
    print("No intersection found in the given range.")